# 🥇 Gold Price Polymarket Analysis

This notebook fetches real-time gold prices and analyzes the Polymarket odds for:
**"What price will gold close at in 2025? ($4000-5000)"**

Market resolves based on COMEX Gold Continuous Contract (GC00) final close price in 2025.

Source: [Polymarket Gold 2025 Market](https://polymarket.com/event/what-price-will-gold-close-at-in-2025-4000-5000)


In [ ]:
# Install required packages
%pip install yfinance pandas numpy matplotlib seaborn scipy -q


Note: you may need to restart the kernel to use updated packages.


## 1. 📊 Fetch Real-Time Gold Price


In [ ]:
# Fetch gold futures data (GC=F is the gold futures ticker on Yahoo Finance)
gold = yf.Ticker("GC=F")

# Get current price
current_data = gold.history(period="5d")
current_price = current_data['Close'].iloc[-1] if len(current_data) > 0 else None

# Get historical data for volatility calculation
hist_data = gold.history(period="1y")

print("═" * 60)
print("🥇 REAL-TIME GOLD PRICE")
print("═" * 60)
print(f"\n📊 Current Gold Futures Price: ${current_price:,.2f}")
print(f"📅 Last Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📈 52-Week High: ${hist_data['High'].max():,.2f}")
print(f"📉 52-Week Low: ${hist_data['Low'].min():,.2f}")


NameError: name 'yf' is not defined

In [ ]:
# Display recent price history with a chart
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(hist_data.index, hist_data['Close'], color='#FFD700', linewidth=2, label='Gold Price')
ax.fill_between(hist_data.index, hist_data['Close'], alpha=0.3, color='#FFD700')
ax.axhline(y=current_price, color='#00FF00', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Current: ${current_price:,.0f}')

# Add bracket lines for Polymarket ranges
bracket_prices = [4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 5000]
for price in bracket_prices:
    ax.axhline(y=price, color='#FF6B6B', linestyle=':', linewidth=0.8, alpha=0.4)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price ($)', fontsize=11)
ax.set_title('📈 Gold Futures (GC=F) - 1 Year Price History', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\n📋 Last 10 Trading Days:")
display(hist_data.tail(10)[['Open', 'High', 'Low', 'Close', 'Volume']].style.format({
    'Open': '${:,.2f}', 'High': '${:,.2f}', 'Low': '${:,.2f}', 
    'Close': '${:,.2f}', 'Volume': '{:,.0f}'
}))


## 2. 🎲 Polymarket Odds (Current)


In [ ]:
# Polymarket odds as of Dec 29, 2025
# Source: https://polymarket.com/event/what-price-will-gold-close-at-in-2025-4000-5000

polymarket_odds = {
    '<$4000': {'prob': 0.004, 'yes_price': 0.004, 'no_price': 0.998, 'volume': 586891},
    '$4000-$4100': {'prob': 0.003, 'yes_price': 0.003, 'no_price': 0.998, 'volume': 323476},
    '$4100-$4200': {'prob': 0.03, 'yes_price': 0.048, 'no_price': 0.992, 'volume': 310980},
    '$4200-$4300': {'prob': 0.04, 'yes_price': 0.046, 'no_price': 0.959, 'volume': 383707},
    '$4300-$4400': {'prob': 0.15, 'yes_price': 0.170, 'no_price': 0.863, 'volume': 354827},
    '$4400-$4500': {'prob': 0.435, 'yes_price': 0.450, 'no_price': 0.581, 'volume': 398667},
    '$4500-$4600': {'prob': 0.21, 'yes_price': 0.220, 'no_price': 0.796, 'volume': 354330},
    '$4600-$4700': {'prob': 0.04, 'yes_price': 0.044, 'no_price': 0.958, 'volume': 391088},
    '$4700-$4800': {'prob': 0.01, 'yes_price': 0.015, 'no_price': 0.990, 'volume': 308167},
    '$4800-$4900': {'prob': 0.01, 'yes_price': 0.007, 'no_price': 0.995, 'volume': 310627},
    '$4900-$5000': {'prob': 0.002, 'yes_price': 0.002, 'no_price': 0.999, 'volume': 388964},
    '>$5000': {'prob': 0.002, 'yes_price': 0.002, 'no_price': 0.999, 'volume': 384525}
}

# Create DataFrame
df_odds = pd.DataFrame(polymarket_odds).T
df_odds.index.name = 'Price Range'
df_odds['prob_pct'] = df_odds['prob'] * 100

print("═" * 60)
print("🎲 POLYMARKET ODDS - Gold 2025 Close")
print("═" * 60)
print("\nTotal Volume: $4,487,038")
print("End Date: December 31, 2025\n")

display(df_odds[['prob_pct', 'yes_price', 'no_price', 'volume']].rename(
    columns={'prob_pct': 'Probability (%)', 'yes_price': 'Yes Price ($)', 
             'no_price': 'No Price ($)', 'volume': 'Volume ($)'}).style.format({
    'Probability (%)': '{:.1f}%',
    'Yes Price ($)': '${:.3f}',
    'No Price ($)': '${:.3f}',
    'Volume ($)': '${:,.0f}'
}).background_gradient(subset=['Probability (%)'], cmap='YlOrRd'))


In [ ]:
# Visualize Polymarket odds
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(df_odds)))
bars = ax.bar(range(len(df_odds)), df_odds['prob_pct'], color=colors, edgecolor='white', linewidth=1.5)

ax.set_xticks(range(len(df_odds)))
ax.set_xticklabels(df_odds.index, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Probability (%)', fontsize=12)
ax.set_xlabel('Price Range', fontsize=12)
ax.set_title('🎯 Polymarket Odds: Gold 2025 Year-End Close', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, prob in zip(bars, df_odds['prob_pct']):
    if prob > 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{prob:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='white')

ax.set_ylim(0, max(df_odds['prob_pct']) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 3. 🔮 Calculate Model-Based Probabilities

Using Monte Carlo simulation with Geometric Brownian Motion to estimate the probability of gold closing in each price bracket.


In [ ]:
# Calculate historical volatility
returns = hist_data['Close'].pct_change().dropna()
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)

# Calculate days until end of 2025
today = datetime.now()
end_2025 = datetime(2025, 12, 31)
days_remaining = max((end_2025 - today).days, 1)
trading_days_remaining = max(int(days_remaining * 252 / 365), 1)

print("═" * 60)
print("📊 VOLATILITY & TIME ANALYSIS")
print("═" * 60)
print(f"\n📈 Historical Volatility:")
print(f"   Daily Volatility: {daily_vol*100:.2f}%")
print(f"   Annualized Volatility: {annual_vol*100:.2f}%")
print(f"\n📅 Time Until Resolution:")
print(f"   Calendar Days Remaining: {days_remaining}")
print(f"   Estimated Trading Days: {trading_days_remaining}")
print(f"\n💰 Current Price: ${current_price:,.2f}")


In [ ]:
# Monte Carlo Simulation for bracket probabilities
def calculate_bracket_probability(current_price, lower, upper, vol, days, drift=0, n_simulations=100000):
    """Monte Carlo simulation for price falling in bracket"""
    dt = days / 252
    
    # Simulate final prices using GBM
    z = np.random.standard_normal(n_simulations)
    final_prices = current_price * np.exp((drift - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z)
    
    # Count prices in bracket
    in_bracket = np.sum((final_prices >= lower) & (final_prices < upper))
    return in_bracket / n_simulations, final_prices

# Define brackets
brackets = [
    ('<$4000', 0, 4000),
    ('$4000-$4100', 4000, 4100),
    ('$4100-$4200', 4100, 4200),
    ('$4200-$4300', 4200, 4300),
    ('$4300-$4400', 4300, 4400),
    ('$4400-$4500', 4400, 4500),
    ('$4500-$4600', 4500, 4600),
    ('$4600-$4700', 4600, 4700),
    ('$4700-$4800', 4700, 4800),
    ('$4800-$4900', 4800, 4900),
    ('$4900-$5000', 4900, 5000),
    ('>$5000', 5000, 100000)
]

# Calculate model probabilities
np.random.seed(42)
model_probs = {}
all_final_prices = None

for name, lower, upper in brackets:
    prob, final_prices = calculate_bracket_probability(
        current_price, lower, upper, annual_vol, trading_days_remaining
    )
    model_probs[name] = prob
    if all_final_prices is None:
        all_final_prices = final_prices

print("═" * 60)
print("🔮 MODEL-BASED PROBABILITIES (Monte Carlo GBM)")
print("═" * 60)
print(f"\nSimulations: 100,000")
print(f"Model: Geometric Brownian Motion (zero drift)\n")

for name, prob in model_probs.items():
    market_prob = polymarket_odds[name]['prob']
    diff = (prob - market_prob) * 100
    indicator = "🟢" if diff > 2 else "🔴" if diff < -2 else "⚪"
    print(f"{indicator} {name}: {prob*100:.2f}% (Market: {market_prob*100:.1f}%, Diff: {diff:+.1f}%)")


## 4. 📊 Market vs Model Comparison


In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Price Range': list(model_probs.keys()),
    'Polymarket Prob (%)': [polymarket_odds[k]['prob']*100 for k in model_probs.keys()],
    'Model Prob (%)': [v*100 for v in model_probs.values()],
    'Yes Price ($)': [polymarket_odds[k]['yes_price'] for k in model_probs.keys()],
    'No Price ($)': [polymarket_odds[k]['no_price'] for k in model_probs.keys()]
})

comparison['Difference (%)'] = comparison['Model Prob (%)'] - comparison['Polymarket Prob (%)']
comparison['Edge (Model/Market)'] = comparison['Model Prob (%)'] / comparison['Polymarket Prob (%)'].replace(0, 0.001)

# Calculate expected value for YES bets
# EV = (prob_win * payout) - (prob_lose * cost)
comparison['EV per $1 YES bet'] = (
    (comparison['Model Prob (%)']/100) * (1 - comparison['Yes Price ($)']) - 
    (1 - comparison['Model Prob (%)']/100) * comparison['Yes Price ($)']
)

# Calculate expected value for NO bets
comparison['EV per $1 NO bet'] = (
    (1 - comparison['Model Prob (%)']/100) * (1 - comparison['No Price ($)']) - 
    (comparison['Model Prob (%)']/100) * comparison['No Price ($)']
)

print("═" * 70)
print("📊 MARKET VS MODEL COMPARISON")
print("═" * 70)

display(comparison.style.background_gradient(subset=['Difference (%)'], cmap='RdYlGn', vmin=-10, vmax=10)
        .background_gradient(subset=['EV per $1 YES bet'], cmap='RdYlGn', vmin=-0.1, vmax=0.1)
        .format({'Polymarket Prob (%)': '{:.2f}%', 'Model Prob (%)': '{:.2f}%', 
                 'Difference (%)': '{:+.2f}%', 'Edge (Model/Market)': '{:.2f}x',
                 'Yes Price ($)': '${:.3f}', 'No Price ($)': '${:.3f}',
                 'EV per $1 YES bet': '${:+.4f}', 'EV per $1 NO bet': '${:+.4f}'}))


In [ ]:
# Visualization: Market vs Model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Side-by-side comparison
x = np.arange(len(comparison))
width = 0.35

ax1 = axes[0]
bars1 = ax1.bar(x - width/2, comparison['Polymarket Prob (%)'], width, label='Polymarket', color='#00D4AA', alpha=0.8)
bars2 = ax1.bar(x + width/2, comparison['Model Prob (%)'], width, label='Model (GBM)', color='#FF6B6B', alpha=0.8)

ax1.set_ylabel('Probability (%)', fontsize=11)
ax1.set_title('📊 Polymarket vs Model Probabilities', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(comparison['Price Range'], rotation=45, ha='right', fontsize=9)
ax1.legend(loc='upper right')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Expected Value
ax2 = axes[1]
colors = ['#00D4AA' if ev > 0 else '#FF6B6B' for ev in comparison['EV per $1 YES bet']]
bars = ax2.bar(x, comparison['EV per $1 YES bet'], color=colors, edgecolor='white', linewidth=1)

ax2.set_ylabel('Expected Value ($)', fontsize=11)
ax2.set_title('💰 Expected Value per $1 YES Bet', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(comparison['Price Range'], rotation=45, ha='right', fontsize=9)
ax2.axhline(y=0, color='white', linestyle='--', alpha=0.5)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 5. 📈 Price Simulation Distribution


In [ ]:
# Plot distribution of simulated final prices
fig, ax = plt.subplots(figsize=(14, 6))

ax.hist(all_final_prices, bins=100, density=True, alpha=0.7, color='#FFD700', edgecolor='white', linewidth=0.5)

# Add bracket boundaries
bracket_bounds = [4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 5000]
for bound in bracket_bounds:
    ax.axvline(x=bound, color='#FF6B6B', linestyle='--', alpha=0.5, linewidth=1)

# Add current price line
ax.axvline(x=current_price, color='#00FF00', linestyle='-', linewidth=2.5, label=f'Current: ${current_price:,.0f}')

# Find most likely bracket based on model
most_likely_idx = comparison['Model Prob (%)'].idxmax()
most_likely = comparison.loc[most_likely_idx, 'Price Range']

ax.set_xlabel('Gold Price ($)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'📈 Simulated Gold Price Distribution at 2025 Year-End\n(Most Likely: {most_likely})', 
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n📊 Simulation Summary Statistics:")
print(f"   Mean Final Price: ${np.mean(all_final_prices):,.2f}")
print(f"   Median Final Price: ${np.median(all_final_prices):,.2f}")
print(f"   Std Dev: ${np.std(all_final_prices):,.2f}")
print(f"   5th Percentile: ${np.percentile(all_final_prices, 5):,.2f}")
print(f"   95th Percentile: ${np.percentile(all_final_prices, 95):,.2f}")


## 6. 🎯 Investment Recommendation


In [ ]:
# Generate investment recommendations
print("═" * 70)
print("🎯 INVESTMENT RECOMMENDATION SUMMARY")
print("═" * 70)
print(f"\n📌 Current Gold Price: ${current_price:,.2f}")
print(f"📌 Days Until Resolution: {days_remaining}")
print(f"📌 Annual Volatility: {annual_vol*100:.1f}%")

# Determine which bracket current price falls into
current_bracket = None
for name, lower, upper in brackets:
    if lower <= current_price < upper:
        current_bracket = name
        break

print(f"📌 Current Price Bracket: {current_bracket}")

# Identify best opportunities
positive_ev_yes = comparison[comparison['EV per $1 YES bet'] > 0.01].sort_values('EV per $1 YES bet', ascending=False)
positive_ev_no = comparison[comparison['EV per $1 NO bet'] > 0.01].sort_values('EV per $1 NO bet', ascending=False)

print("\n" + "═" * 70)
print("✅ POSITIVE EV OPPORTUNITIES - YES BETS")
print("═" * 70)

if len(positive_ev_yes) > 0:
    for idx, row in positive_ev_yes.head(5).iterrows():
        edge = (row['Model Prob (%)'] / row['Polymarket Prob (%)'] - 1) * 100
        print(f"\n🟢 {row['Price Range']}")
        print(f"   Market Probability: {row['Polymarket Prob (%)']:.1f}%")
        print(f"   Model Probability: {row['Model Prob (%)']:.1f}%")
        print(f"   Edge over Market: {edge:+.1f}%")
        print(f"   Yes Price: ${row['Yes Price ($)']:.3f}")
        print(f"   Expected Value: ${row['EV per $1 YES bet']:+.4f} per $1 bet")
        
        # Kelly Criterion for position sizing
        p = row['Model Prob (%)']/100
        if row['Yes Price ($)'] > 0:
            b = (1 - row['Yes Price ($)']) / row['Yes Price ($)']  # odds ratio
            kelly = max(0, (p * b - (1-p)) / b) if b > 0 else 0
            print(f"   Kelly Fraction: {kelly*100:.1f}% of bankroll")
else:
    print("\n⚠️ No positive EV YES opportunities identified based on the model.")

print("\n" + "═" * 70)
print("✅ POSITIVE EV OPPORTUNITIES - NO BETS")
print("═" * 70)

if len(positive_ev_no) > 0:
    for idx, row in positive_ev_no.head(3).iterrows():
        print(f"\n🔵 {row['Price Range']} - BUY NO")
        print(f"   Model says {row['Model Prob (%)']:.1f}% (Market: {row['Polymarket Prob (%)']:.1f}%)")
        print(f"   No Price: ${row['No Price ($)']:.3f}")
        print(f"   Expected Value: ${row['EV per $1 NO bet']:+.4f} per $1 bet")
else:
    print("\n⚠️ No significant NO bet opportunities identified.")


In [ ]:
# Final Summary Table with Recommendations
print("\n" + "═" * 70)
print("📋 FINAL RECOMMENDATION TABLE")
print("═" * 70)

recommendation_df = comparison[['Price Range', 'Polymarket Prob (%)', 'Model Prob (%)', 
                                  'Yes Price ($)', 'No Price ($)', 'EV per $1 YES bet', 'EV per $1 NO bet']].copy()

def get_recommendation(row):
    ev_yes = row['EV per $1 YES bet']
    ev_no = row['EV per $1 NO bet']
    
    if ev_yes > 0.05:
        return '🟢 STRONG BUY YES'
    elif ev_yes > 0.02:
        return '🟡 BUY YES'
    elif ev_no > 0.05:
        return '🔵 STRONG BUY NO'
    elif ev_no > 0.02:
        return '🔵 BUY NO'
    elif ev_yes < -0.05:
        return '🔴 AVOID YES'
    else:
        return '⚪ NEUTRAL'

recommendation_df['Recommendation'] = recommendation_df.apply(get_recommendation, axis=1)

display(recommendation_df.style.format({
    'Polymarket Prob (%)': '{:.1f}%',
    'Model Prob (%)': '{:.1f}%',
    'Yes Price ($)': '${:.3f}',
    'No Price ($)': '${:.3f}',
    'EV per $1 YES bet': '${:+.4f}',
    'EV per $1 NO bet': '${:+.4f}'
}).background_gradient(subset=['EV per $1 YES bet'], cmap='RdYlGn', vmin=-0.1, vmax=0.1))


In [ ]:
# Key Takeaways & Disclaimer
print("\n" + "═" * 70)
print("🎯 KEY TAKEAWAYS")
print("═" * 70)

# Find best opportunity
best_yes = comparison.loc[comparison['EV per $1 YES bet'].idxmax()]
best_no = comparison.loc[comparison['EV per $1 NO bet'].idxmax()]

print(f"""
📊 MARKET ASSESSMENT:
   • Current gold price (${current_price:,.2f}) suggests the market expects 
     gold to close near ${current_price:,.0f} by end of 2025
   • Most likely bracket per Polymarket: $4400-$4500 (43.5%)
   • Most likely bracket per Model: {comparison.loc[comparison['Model Prob (%)'].idxmax(), 'Price Range']} ({comparison['Model Prob (%)'].max():.1f}%)

💡 BEST OPPORTUNITIES:
   • Best YES bet: {best_yes['Price Range']} 
     (EV: ${best_yes['EV per $1 YES bet']:+.4f}/$ at ${best_yes['Yes Price ($)']:.3f})
   • Best NO bet: {best_no['Price Range']} 
     (EV: ${best_no['EV per $1 NO bet']:+.4f}/$ at ${best_no['No Price ($)']:.3f})

⏰ TIME FACTOR:
   • With only {days_remaining} days remaining, volatility impact is limited
   • Prices are likely to stay close to current levels
   • Near-money brackets have highest probability density
""")

print("═" * 70)
print("⚠️ RISK DISCLAIMER")
print("═" * 70)
print("""
This analysis is for INFORMATIONAL PURPOSES ONLY and should NOT be 
considered financial advice. Key limitations:

1. MODEL ASSUMPTIONS:
   • Assumes random walk with constant volatility (GBM)
   • Uses historical volatility which may not predict future moves
   • Assumes zero drift (no directional bias)

2. FACTORS NOT CONSIDERED:
   • Macroeconomic events (Fed policy, inflation data)
   • Geopolitical risks and safe-haven flows
   • Gold-specific supply/demand factors
   • Year-end rebalancing and liquidity effects
   • Market microstructure and slippage

3. PREDICTION MARKET CONSIDERATIONS:
   • Market may have information not captured by the model
   • Liquidity varies across brackets
   • Spreads can significantly impact actual returns

⚠️ IMPORTANT: Always do your own research and only invest what you can 
afford to lose. Past performance is not indicative of future results.
""")


---

## 📚 References

- **Polymarket**: [Gold 2025 Close Market](https://polymarket.com/event/what-price-will-gold-close-at-in-2025-4000-5000)
- **Resolution Source**: COMEX Gold Continuous Contract (GC00) - [Google Finance](https://www.google.com/finance/quote/GCW00:COMEX)
- **Data Source**: Yahoo Finance (GC=F ticker)
- **Model**: Geometric Brownian Motion with Monte Carlo simulation (100,000 paths)

---

*Analysis generated dynamically - re-run notebook for updated prices and recommendations*


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats

# Set style for beautiful visualizations
plt.style.use('dark_background')
sns.set_palette('viridis')
pd.set_option('display.max_columns', None)
